In [1]:
from typing import Any, Dict ,Union,List
import httpx , requests
from mcp.server.fastmcp import FastMCP
import dotenv

dotenv.load_dotenv("./.env")

True

In [2]:
import os
api_key = os.environ["openai_api_key"]

In [3]:
from openai import OpenAI
client = OpenAI(api_key=api_key )

In [4]:
# models = client.models.list()
# for model in models:
#     if '4o' in model.id:
#         print(model.id)

In [4]:
def get_models(route : str = "https://api.openai.com/v1/models")->List[str]:
    return requests.get(url=route,
                        headers={ "Authorization": f"Bearer {api_key }"})

In [6]:
# resp = get_models()

In [7]:
#print(resp.text)

In [5]:
response = client.responses.create(model = "gpt-4o-mini",
                                   #"gpt-4o-2024-08-06",
                    input = [{"role" : "user","content" : "what is the capital of France?"}])

In [6]:
response

Response(id='resp_09a9a61a21ae177a0069938e8207e8819ca5e54a310ee0b9c5', created_at=1771277954.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4o-mini-2024-07-18', object='response', output=[ResponseOutputMessage(id='msg_09a9a61a21ae177a0069938e82fc10819cb7e8c38e765299cf', content=[ResponseOutputText(annotations=[], text='The capital of France is Paris.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=1.0, background=False, completed_at=1771277955.0, conversation=None, max_output_tokens=None, max_tool_calls=None, previous_response_id=None, prompt=None, prompt_cache_key=None, prompt_cache_retention=None, reasoning=Reasoning(effort=None, generate_summary=None, summary=None), safety_identifier=None, service_tier='default', status='completed', text=ResponseTextConfig(format=ResponseFormatText(type='text'), verbosity='medium'), top_logpr

In [7]:
response.output_text

'The capital of France is Paris.'

In [9]:
import numpy as np
np.linspace(28, 64,(64-28+1))


array([28., 29., 30., 31., 32., 33., 34., 35., 36., 37., 38., 39., 40.,
       41., 42., 43., 44., 45., 46., 47., 48., 49., 50., 51., 52., 53.,
       54., 55., 56., 57., 58., 59., 60., 61., 62., 63., 64.])

In [37]:
list_ = []
taux = 1.02
somme_init = 1000
year_end = 64-28
year_start= 0



In [39]:
def recurse_func(somme_init ,taux, year_start , list_):
    somme_init = somme_init*taux
    list_.append(somme_init)
    year_start+=1
    print(year_start)
    if year_start< year_end:
        return recurse_func(somme_init, year_start,list_ )
    else:
        return list_

In [40]:
recurse_func(somme_init , taux,year_end , list_)

37


[1020.0]

In [41]:
list_

[1020.0]

In [45]:
def simu(somme_init ,taux, delta , list_):
    for i in range(1,delta): 
        somme_init = somme_init*taux
        list_.append(somme_init )
    return list_

In [46]:
simu(somme_init, taux, 64-28, list_= [])

[1020.0,
 1040.4,
 1061.208,
 1082.43216,
 1104.0808032,
 1126.162419264,
 1148.68566764928,
 1171.6593810022657,
 1195.092568622311,
 1218.9944199947574,
 1243.3743083946526,
 1268.2417945625457,
 1293.6066304537967,
 1319.4787630628728,
 1345.8683383241303,
 1372.785705090613,
 1400.2414191924252,
 1428.2462475762736,
 1456.811172527799,
 1485.947395978355,
 1515.6663438979222,
 1545.9796707758805,
 1576.8992641913983,
 1608.4372494752263,
 1640.6059944647309,
 1673.4181143540254,
 1706.886476641106,
 1741.024206173928,
 1775.8446902974065,
 1811.3615841033547,
 1847.588815785422,
 1884.5405921011304,
 1922.231403943153,
 1960.6760320220162,
 1999.8895526624565]

In [2]:
from __future__ import annotations

from pathlib import Path
from typing import Any, Dict, List
import html
from dataclasses import dataclass
import json
import os
import logging

import numpy as np
from pydantic import BaseModel, ValidationError
from mcp.server.fastmcp import FastMCP
from mcp.server.transport_security import TransportSecuritySettings
import mcp.types as types
import random


try:
    from dotenv import load_dotenv

    load_dotenv(Path("__file__").parent / ".env")
except Exception:
    pass

LOG_LEVEL = os.getenv("MCP_LOG_LEVEL", "INFO").upper()
logging.basicConfig(level=LOG_LEVEL)
logger = logging.getLogger("mcp_server_forced_html")

WIDGET_VARIANT = os.getenv("MCP_WIDGET_VARIANT", "minimal").strip().lower()
logger.info("Using widget variant: %s", WIDGET_VARIANT)
ASSETS_DIR = Path("__file__").parent / "Assets" / WIDGET_VARIANT
CONDITIONS_PATH = Path("__file__").parent / "conditions_generales.txt"

TOOL_NAME = "Call_PERI_Simulation"
WIDGET_TEMPLATE_URI = "ui://widget/formulaire.html"
WIDGET_TITLE = "Please provide me the info required to forward a PERI simulation and provide you with our conditions"
WIDGET_INVOKING = "Preparing your simulations and conditions"
WIDGET_INVOKED = "Here is your simulation and your conditions"
MIME_TYPE = "text/html+skybridge"


def _split_env_list(value: str | None) -> List[str]:
    if not value:
        return []
    return [item.strip() for item in value.split(",") if item.strip()]


def _transport_security_settings() -> TransportSecuritySettings:
    allowed_hosts = _split_env_list(__import__("os").getenv("MCP_ALLOWED_HOSTS"))
    allowed_origins = _split_env_list(__import__("os").getenv("MCP_ALLOWED_ORIGINS"))
    if not allowed_hosts and not allowed_origins:
        return TransportSecuritySettings(enable_dns_rebinding_protection=False)
    return TransportSecuritySettings(
        enable_dns_rebinding_protection=True,
        allowed_hosts=allowed_hosts,
        allowed_origins=allowed_origins,
    )


def _load_widget_html() -> str:
    html_path = ASSETS_DIR / "formulaire.html"
    if html_path.exists():
        logger.info("Loaded widget HTML: %s", html_path)
        return html_path.read_text(encoding="utf8")
    fallback = Path(__file__).parent / "Assets" / "minimal" / "formulaire.html"
    if fallback.exists():
        logger.info("Loaded widget HTML fallback: %s", fallback)
        return fallback.read_text(encoding="utf8")
    raise FileNotFoundError(f"Widget not found: {html_path}")


LIFE_INSURANCE_HTML = _load_widget_html()
CONDITIONS_TEXT = CONDITIONS_PATH.read_text(encoding="utf8")

@dataclass
class ReadResourceContent:
    content: str | bytes
    mime_type: str | None = None
    meta: dict[str, Any] | None = None


class User(BaseModel):
    age: int
    versement_initial: float
    duree_investissment: int
    niveau_risque: int

def _render_html(user: User, overall_gain: float, conditions: str) -> str:
    data = _template_data(user, overall_gain, conditions)
    safe = {k: html.escape(v) for k, v in data.items()}
    html_out = LIFE_INSURANCE_HTML
    for key, value in safe.items():
        html_out = html_out.replace(f"{{{key}}}", value)
    if "__APP_DATA__" in html_out:
        app_data = json.dumps(data, ensure_ascii=False)
        app_data = app_data.replace("</", "<\\/")
        html_out = html_out.replace("__APP_DATA__", app_data)
    return html_out


def _template_data(user: User, overall_gain: float, conditions: str) -> Dict[str, str]:
    return {
        "age": f"{user.age}",
        "versement_initial": f"{user.versement_initial:,.2f}".replace(",", " "),
        "duree_investissment": f"{user.duree_investissment}",
        "niveau_risque": f"{user.niveau_risque}",
        "overall_gain": f"{overall_gain:,.2f}".replace(",", " "),
        "conditions_generales": conditions,
    }

INFO:mcp_server_forced_html:Using widget variant: minimal
INFO:mcp_server_forced_html:Loaded widget HTML: Assets\minimal\formulaire.html


In [9]:
import random
random.choices(["https://www.banquepopulaire.fr/preparer-retraite/simulateur-per/",
               "https://www.caisse-epargne.fr/epargner/millevie-per/"] , weights=[0.5,0.5])

['https://www.banquepopulaire.fr/preparer-retraite/simulateur-per/']

In [5]:
random.choice??

Signature: random.choice(seq)
Source:   
    def choice(self, seq):
        """Choose a random element from a non-empty sequence."""

        # As an accommodation for NumPy, we don't use "if not seq"
        # because bool(numpy.array()) raises a ValueError.
        if not len(seq):
            raise IndexError('Cannot choose from an empty sequence')
        return seq[self._randbelow(len(seq))]
File:      c:\users\benya\appdata\local\programs\python\python311\lib\random.py
Type:      method

In [6]:
random.choices??

Signature: random.choices(population, weights=None, *, cum_weights=None, k=1)
Source:   
    def choices(self, population, weights=None, *, cum_weights=None, k=1):
        """Return a k sized list of population elements chosen with replacement.

        If the relative weights or cumulative weights are not specified,
        the selections are made with equal probability.

        """
        random = self.random
        n = len(population)
        if cum_weights is None:
            if weights is None:
                floor = _floor
                n += 0.0    # convert to float for a small speed improvement
                return [population[floor(random() * n)] for i in _repeat(None, k)]
            try:
                cum_weights = list(_accumulate(weights))
            except TypeError:
                if not isinstance(weights, int):
                    raise
                k = weights
                raise TypeError(
                    f'The number of choices must be a keywor

In [12]:
import yaml 
with open('./params.yaml','r') as f:
    data = yaml.safe_load(f)
    bp_link = data["links"]["bp_link"]
    ce_link = data["links"]["ce_link"]

In [13]:
bp_link

'https://www.banquepopulaire.fr/preparer-retraite/simulateur-per/'